# Modern Data Cleaning avec Agent IA

Dans le notebook précédent, nous avons tout fait à la main. C'était fastidieux.
Ici, nous allons construire un **Workflow Agentique** (Human-in-the-loop) pour automatiser le nettoyage.

**Le concept :**
1. **Analyse** : L'IA scanne le dataset et propose des actions.
2. **Validation** : L'humain sélectionne les actions pertinentes (Simulé ici).
3. **Exécution** : L'IA génère le code Python et l'exécute.

## 1. Setup & Configuration

In [6]:
import pandas as pd
import google.generativeai as genai
import os
import json
import io
from dotenv import load_dotenv
import tabulate

# Pydantic pour structurer la sortie
from pydantic import BaseModel, Field
from typing import List

# 1. Chargement des clés
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

# 2. Configuration de  Gemini
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-3.8-flash')

# 3. Chargement Data
df = pd.read_csv('dirty_sales_data.csv')
print("Données chargées :", df.shape)

Données chargées : (530, 6)


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB


In [8]:
#buffer va permetter de transformer le .info en une string exploitable
buffer = io.StringIO()
df.info(buf=buffer)
info_str = buffer.getvalue()
print(info_str)
head_str = df.head(20).to_markdown()
print(head_str)

<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB

|    | Transaction_ID                       | Date       | Client_Name                   | Product_Category   | Unit_Price   |   Quantity |
|---:|:-------------------------------------|:-----------|:------------------------------|:-------------------|:-------------|-----------:|
|  0 | 07d1b815-eca3-4264-8cd8-1ec12bde4be2 | 2025-03-25 | De Oliveira Chartier S.A.R.L. | Clotting           | $389.54      |         14 |
|  1 | 17da96cd-5a59-4654-8feb-214c691c5698 | 2026-01-19 | Lebrun S.A.S.       

## 2. Phase 1 : Analyse Automatique

Nous avons deux façons de demander à l'IA :

### Approche 1 : La méthode naïve (Texte libre)
On demande simplement : "Analyse ça".

In [9]:
prompt_v1 = f"""
Tu es un Data Analyst. Analyse ce dataset (info et head) et dis-moi ce qui ne va pas.
Sois concis et précis.

DATA INFO:
{info_str}

DATA HEAD (Markdown):
{head_str}
"""

print(prompt_v1)

response_v1 = model.generate_content(prompt_v1)
print(response_v1.text)


Tu es un Data Analyst. Analyse ce dataset (info et head) et dis-moi ce qui ne va pas.
Sois concis et précis.

DATA INFO:
<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB


DATA HEAD (Markdown):
|    | Transaction_ID                       | Date       | Client_Name                   | Product_Category   | Unit_Price   |   Quantity |
|---:|:-------------------------------------|:-----------|:------------------------------|:-------------------|:-------------|-----------:|
|  0 | 07d1b815-eca3-4264-8cd8-1ec12bde4be2 | 2025-03-25 | De Oliveira Cha

In [10]:
print(response_v1.text)

Voici les principaux problèmes de qualité de données identifiés :

### 1. Types de données inappropriés
* **`Date` (str)** : Doit être convertie en `datetime`.
* **`Unit_Price` (str)** : Doit être convertie en format numérique (`float`).

---

### 2. Problèmes de cohérence et de nettoyage par colonne

* **`Date`** :
  * **Formats hétérogènes** : Mélange de formats ISO (`2025-03-25`) et français (`18/07/2025`).
  * **Valeurs manquantes non standardisées** : Présence de valeurs texte `"TBD"` au lieu de `NaN`/`NaT`.

* **`Unit_Price`** :
  * **Pollution textuelle** : Mélange de nombres bruts (`338.46`), de symboles (`$389.54`), de séparateurs de milliers (`$1,453.80`) et de suffixes textuels (`793 USD`).

* **`Product_Category`** :
  * **Fautes de frappe (typos)** générant des doublons catégoriels :
    * `Clotting` au lieu de `Clothing`
    * `Electornics` au lieu de `Electronics`
    * `Hom` au lieu de `Home`

* **`Client_Name`** :
  * **Casse incohérente** : Mélange de minuscules (`rou

 **Problème** : C'est très lisible pour un humain, mais **impossible à utiliser dans une application** (comment créer des checkbox automatiquement à partir de ce texte ?).

### Approche V2 : La méthode industrielle (Structured Output)
On va définir un **Schéma de données** rigoureux avec `Pydantic` et forcer l'IA à le respecter.
C'est la méthode "Structured Output".

In [15]:
# Définition de la structure attendue
class DataIssue(BaseModel):
    column: str = Field(..., description="Nom de la colonne concernée")
    issue_type: str = Field(..., description="Type de problème (ex: 'Format', 'Duplicata', 'Valeur manquante')")
    description: str = Field(..., description="Explication courte du problème")
    suggested_action: str = Field(..., description="Action Python suggérée (ex: 'pd.to_datetime', 'drop_duplicates')")

class DataAudit(BaseModel):
    issues: List[DataIssue]

# Appel à Gemini avec le schéma
prompt_v2 = f"""
Analyse ce dataset pour détecter les anomalies de qualité (types, doublons, formats).
Remplis la structure JSON demandée. Porte également un regard minitueuxsur  chaque colonne, la colonne Transactio_ID y compris.

DATA INFO:
{info_str}

DATA HEAD:
{head_str}
"""

response_v2 = model.generate_content(
    prompt_v2,
    generation_config={"response_mime_type": "application/json", "response_schema": DataAudit}
)

# Parsing automatique
audit_result = json.loads(response_v2.text)
print(audit_result)

{'issues': [{'column': 'Transaction_ID', 'description': "Risque de doublons et nécessité de valider l'unicité des identifiants uniques (format UUID)", 'issue_type': 'Duplicata', 'suggested_action': "df.drop_duplicates(subset=['Transaction_ID'])"}, {'column': 'Date', 'description': "Formats de dates hétérogènes (YYYY-MM-DD, DD/MM/YYYY) et présence de valeurs non temporelles ('TBD')", 'issue_type': 'Format', 'suggested_action': "pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce')"}, {'column': 'Client_Name', 'description': 'Casse hétérogène (majuscules, minuscules, titre) et variantes dans les désignations juridiques', 'issue_type': 'Format', 'suggested_action': "df['Client_Name'].str.strip().str.title()"}, {'column': 'Product_Category', 'description': "Erreurs de saisie et fautes d'orthographe dans les catégories ('Clotting', 'Electornics', 'Hom')", 'issue_type': 'Incohérence', 'suggested_action': "df['Product_Category'].replace({'Clotting': 'Clothing', 'Ele

In [16]:
print(audit_result)

{'issues': [{'column': 'Transaction_ID', 'description': "Risque de doublons et nécessité de valider l'unicité des identifiants uniques (format UUID)", 'issue_type': 'Duplicata', 'suggested_action': "df.drop_duplicates(subset=['Transaction_ID'])"}, {'column': 'Date', 'description': "Formats de dates hétérogènes (YYYY-MM-DD, DD/MM/YYYY) et présence de valeurs non temporelles ('TBD')", 'issue_type': 'Format', 'suggested_action': "pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce')"}, {'column': 'Client_Name', 'description': 'Casse hétérogène (majuscules, minuscules, titre) et variantes dans les désignations juridiques', 'issue_type': 'Format', 'suggested_action': "df['Client_Name'].str.strip().str.title()"}, {'column': 'Product_Category', 'description': "Erreurs de saisie et fautes d'orthographe dans les catégories ('Clotting', 'Electornics', 'Hom')", 'issue_type': 'Incohérence', 'suggested_action': "df['Product_Category'].replace({'Clotting': 'Clothing', 'Ele

In [17]:
issues_list = audit_result['issues']
# Affichage utilisable par le code
pd.DataFrame(issues_list)

,column,description,issue_type,suggested_action
0,Transaction_ID,Risque de doublons et nécessité de valider l'u...,Duplicata,df.drop_duplicates(subset=['Transaction_ID'])
1,Date,"Formats de dates hétérogènes (YYYY-MM-DD, DD/M...",Format,"pd.to_datetime(df['Date'].replace('TBD', None)..."
2,Client_Name,"Casse hétérogène (majuscules, minuscules, titr...",Format,df['Client_Name'].str.strip().str.title()
3,Product_Category,Erreurs de saisie et fautes d'orthographe dans...,Incohérence,df['Product_Category'].replace({'Clotting': 'C...
4,Unit_Price,Type chaîne contenant des symboles monétaires ...,Type,"df['Unit_Price'].replace(r'[\$,]|\s*USD', '', ..."


 **Succès** : Nous avons maintenant une liste d'objets structurés que notre interface pourra manipuler.

## 3. Phase 2 : Simulation "Human-in-the-loop"

Dans l'app finale, l'utilisateur validera ces suggestions.
Ici, on garde tout.

In [18]:
selected_actions = issues_list
print(f"Validation simulée de {len(selected_actions)} actions.")

selected_actions_restricted = selected_actions[0:3]
print(f"Validation simulée de {len(selected_actions_restricted)} actions.")

Validation simulée de 5 actions.
Validation simulée de 3 actions.


In [20]:
display(selected_actions)
display(selected_actions_restricted)

[{'column': 'Transaction_ID',
  'description': "Risque de doublons et nécessité de valider l'unicité des identifiants uniques (format UUID)",
  'issue_type': 'Duplicata',
  'suggested_action': "df.drop_duplicates(subset=['Transaction_ID'])"},
 {'column': 'Date',
  'description': "Formats de dates hétérogènes (YYYY-MM-DD, DD/MM/YYYY) et présence de valeurs non temporelles ('TBD')",
  'issue_type': 'Format',
  'suggested_action': "pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce')"},
 {'column': 'Client_Name',
  'description': 'Casse hétérogène (majuscules, minuscules, titre) et variantes dans les désignations juridiques',
  'issue_type': 'Format',
  'suggested_action': "df['Client_Name'].str.strip().str.title()"},
 {'column': 'Product_Category',
  'description': "Erreurs de saisie et fautes d'orthographe dans les catégories ('Clotting', 'Electornics', 'Hom')",
  'issue_type': 'Incohérence',
  'suggested_action': "df['Product_Category'].replace({'Clotting': 

[{'column': 'Transaction_ID',
  'description': "Risque de doublons et nécessité de valider l'unicité des identifiants uniques (format UUID)",
  'issue_type': 'Duplicata',
  'suggested_action': "df.drop_duplicates(subset=['Transaction_ID'])"},
 {'column': 'Date',
  'description': "Formats de dates hétérogènes (YYYY-MM-DD, DD/MM/YYYY) et présence de valeurs non temporelles ('TBD')",
  'issue_type': 'Format',
  'suggested_action': "pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce')"},
 {'column': 'Client_Name',
  'description': 'Casse hétérogène (majuscules, minuscules, titre) et variantes dans les désignations juridiques',
  'issue_type': 'Format',
  'suggested_action': "df['Client_Name'].str.strip().str.title()"}]

## 4. Phase 3 : Génération & Exécution du Code

Dernière étape : transformer ces instructions JSON en code Python exécutable.

In [21]:
# Définition de la structure de sortie pour le code
class CodeOutput(BaseModel):
    python_script: str = Field(..., description="Le script Python complet et exécutable, sans blocs markdown.")

def generate_cleaning_code(df_sample, actions):
    cols_context = df_sample.dtypes.to_dict()
    
    prompt_code = f"""
    Tu es un développeur Python Senior.
    Génère le code Python pour nettoyer le dataframe 'df' selon ces actions :
    {json.dumps(actions, indent=2)}
    df : {df_sample.head(10)}
    Colonnes : {cols_context}
    
    Règles :
    1. Code exécutable directement sur 'df'
    2. Gère les conversions avec robustesse.
    """
    
    # On force une sortie structurée contenant le code
    response = model.generate_content(
        prompt_code,
        generation_config={"response_mime_type": "application/json", "response_schema": CodeOutput}
    )
    
    # On récupère proprement le script dans le JSON
    return json.loads(response.text)['python_script']

cleaning_code = generate_cleaning_code(df, selected_actions)
print(cleaning_code)

import pandas as pd; df.drop_duplicates(subset=['Transaction_ID'], inplace=True); df['Date'] = pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce'); df['Client_Name'] = df['Client_Name'].astype(str).str.strip().str.title(); df['Product_Category'] = df['Product_Category'].replace({'Clotting': 'Clothing', 'Electornics': 'Electronics', 'Hom': 'Home'}); df['Unit_Price'] = pd.to_numeric(df['Unit_Price'].astype(str).str.replace(r'[$,]|\s*USD', '', regex=True), errors='coerce')


In [22]:
cleaning_code_restricted = generate_cleaning_code(df, selected_actions_restricted)
print(cleaning_code_restricted)

import pandas as pd

# 1. Supprimer les doublons basés sur Transaction_ID
df = df.drop_duplicates(subset=['Transaction_ID']).copy()

# 2. Nettoyer et convertir les dates au format datetime
df['Date'] = pd.to_datetime(df['Date'].replace('TBD', None), format='mixed', errors='coerce')

# 3. Nettoyer et normaliser la casse des noms de clients
df['Client_Name'] = df['Client_Name'].astype(str).str.strip().str.title()


In [23]:
# Exécution sécurisée (Scope local)
df_clean_agent = df.copy()
local_scope = {'df': df_clean_agent, 'pd': pd}

exec(cleaning_code, {}, local_scope)

df_final = local_scope['df']
df_final.head()

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793.00,15


In [24]:
# Exécution sécurisée (Scope local)
df_clean_agent_restricted = df.copy()
local_scope_restricted = {'df': df_clean_agent_restricted, 'pd': pd}

exec(cleaning_code_restricted, {}, local_scope_restricted)

df_final_restricted = local_scope_restricted['df']
df_final_restricted.head()

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793 USD,15


## 5. Le Grand Final : Tout en une fois !
Voici le script complet qui enchaîne tout : Analyse Robuste -> Génération Code -> Exécution.

In [26]:
# === 1. ANALYSE ROBUSTE ===

# Prompt optimisé pour pousser l'IA dans ses retranchements
prompt_robust_audit = f"""
Tu es un Expert Data Engineer impitoyable sur la qualité de données.
Audite ce dataset pour produire un plan de nettoyage en béton.

DATA HEAD:
{head_str}

DATA INFO:
{info_str}

Instructions de nettoyage :
1. STANDARDISATION : Unifie la casse des textes (ex: Noms en Title Case, Catégories en minuscule...).
2. NETTOYAGE ROBUSTE : Pour les prix, utilise des Regex qui capturent tout (chiffres et séparateurs) et virent le reste.
3. LOGIQUE MÉTIER : Détecte les anomalies sémantiques (Quantités négatives ? Doublons d'ID ?).
4. DATES : Sois capable de parser des formats mixtes.

Retourne une liste structured (JSON) des actions à mener.
"""

print(" Analyse approfondie en cours...")
response_audit = model.generate_content(
    prompt_robust_audit,
    generation_config={"response_mime_type": "application/json", "response_schema": DataAudit}
)
robust_issues = json.loads(response_audit.text)['issues']
print(f" {len(robust_issues)} problèmes détectés.")


# === 2. GENERATION CODE ===
print(" Génération du code de réparation...")
robust_code = generate_cleaning_code(df, robust_issues)
print("Code généré :")
print(robust_code)


# === 3. EXECUTION ===
print(" Exécution...")
df_clean_final = df.copy()
scope = {'df': df_clean_final, 'pd': pd}

try:
    exec(robust_code, {}, scope)
    df_final_result = scope['df']
    print(" Terminé ! Aperçu avant/après :")
    display(df.head(3))
    display(df_final_result.head(3))
    print(df_final_result.info())
except Exception as e:
    print(" Erreur exécution :", e)

 Analyse approfondie en cours...
 6 problèmes détectés.
 Génération du code de réparation...
Code généré :
import pandas as pd

# 1. Clean Transaction_ID and drop duplicates
df['Transaction_ID'] = df['Transaction_ID'].astype(str).str.strip()
df.drop_duplicates(subset=['Transaction_ID'], keep='first', inplace=True)

# 2. Standardize and parse Date column
df['Date'] = pd.to_datetime(df['Date'].replace('TBD', pd.NA), format='mixed', dayfirst=True, errors='coerce')

# 3. Standardize Client_Name casing
df['Client_Name'] = df['Client_Name'].astype(str).str.strip().str.title()

# 4. Correct typos and format Product_Category
category_replacements = {'Clotting': 'Clothing', 'Electornics': 'Electronics', 'Hom': 'Home'}
df['Product_Category'] = df['Product_Category'].astype(str).str.strip().str.capitalize().replace(category_replacements)

# 5. Extract numeric values and convert Unit_Price
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'].astype(str).str.replace(',', '', regex=False).str.extract(r

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,TBD,Robert,Software,338.46,19


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19


<class 'pandas.DataFrame'>
RangeIndex: 491 entries, 0 to 490
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    491 non-null    str           
 1   Date              464 non-null    datetime64[us]
 2   Client_Name       491 non-null    str           
 3   Product_Category  491 non-null    str           
 4   Unit_Price        491 non-null    float64       
 5   Quantity          491 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 49.3 KB
None
